<a href="https://colab.research.google.com/github/Asritha0507/ML-Market-Basket-Analysis/blob/main/10_Hybrid_Association_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# RECOVERY CELL — IMPORT LIBRARIES AND LOAD REQUIRED DATA
# ============================================================

import pandas as pd
import numpy as np
import os
import gc

from collections import Counter

BASE_PATH = "/content/drive/MyDrive/ML_Market_Basket_Analysis"
DATA_PATH = os.path.join(BASE_PATH, "Datasets")
MODEL_PATH = os.path.join(BASE_PATH, "Models", "Model_Outputs")

ORDERS_PATH = os.path.join(DATA_PATH, "orders.csv")
PRIOR_PATH = os.path.join(DATA_PATH, "order_products_prior.csv")

TRAIN_ORDER_MAX = 50
SAMPLE_SIZE = 50000
RANDOM_STATE = 42

orders = pd.read_csv(ORDERS_PATH)

train_orders = orders[
    orders["order_number"] <= TRAIN_ORDER_MAX
].copy()

prior = pd.read_csv(
    PRIOR_PATH,
    usecols=["order_id", "product_id"]
)

train_order_ids = set(train_orders["order_id"])

prior_train = prior[
    prior["order_id"].isin(train_order_ids)
].copy()

baskets = (
    prior_train
    .groupby("order_id")["product_id"]
    .apply(lambda x: list(set(x)))
    .tolist()
)

np.random.seed(RANDOM_STATE)

sampled_indices = np.random.choice(
    len(baskets),
    size=min(SAMPLE_SIZE, len(baskets)),
    replace=False
)

sampled_baskets = [
    baskets[i] for i in sampled_indices
]

print("Recovery completed.")
print("Training baskets:", len(baskets))
print("Sampled baskets:", len(sampled_baskets))
print("Random state:", RANDOM_STATE)

Recovery completed.
Training baskets: 3008914
Sampled baskets: 50000
Random state: 42


In [2]:
# ============================================================
# CELL 2 — PREPARE MEMORY-EFFICIENT FP-GROWTH DATA
# ============================================================

FP_SAMPLE_SIZE = 20000
FP_MIN_SUPPORT = 0.001

fp_baskets = sampled_baskets[:FP_SAMPLE_SIZE]

# Count product occurrences
product_counts = Counter()

for basket in fp_baskets:
    product_counts.update(set(basket))

min_count = int(FP_MIN_SUPPORT * len(fp_baskets))

frequent_products = {
    product
    for product, count in product_counts.items()
    if count >= min_count
}

# Keep only sufficiently frequent products
fp_baskets_filtered = [
    [product for product in basket if product in frequent_products]
    for basket in fp_baskets
]

fp_baskets_filtered = [
    basket for basket in fp_baskets_filtered
    if basket
]

print("FP-Growth data prepared.")
print("Baskets used:", len(fp_baskets_filtered))
print("Products before filtering:", len(product_counts))
print("Minimum product count:", min_count)
print("Products retained:", len(frequent_products))

FP-Growth data prepared.
Baskets used: 19071
Products before filtering: 21663
Minimum product count: 20
Products retained: 1883


In [3]:
# ============================================================
# CELL 3 — FP-GROWTH FREQUENT ITEMSET MINING
# ============================================================

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth

encoder_fp = TransactionEncoder()

fp_array = encoder_fp.fit(
    fp_baskets_filtered
).transform(
    fp_baskets_filtered
)

fp_basket_df = pd.DataFrame(
    fp_array,
    columns=encoder_fp.columns_
)

print("FP-Growth encoding completed.")
print("Encoded shape:", fp_basket_df.shape)

print("\nRunning FP-Growth...")

frequent_itemsets_fpgrowth = fpgrowth(
    fp_basket_df,
    min_support=FP_MIN_SUPPORT,
    use_colnames=True,
    max_len=2
)

print("\nFP-Growth completed.")
print("Number of frequent itemsets:",
      len(frequent_itemsets_fpgrowth))

print("\nItemset size distribution:")
print(
    frequent_itemsets_fpgrowth["itemsets"]
    .apply(len)
    .value_counts()
    .sort_index()
)

FP-Growth encoding completed.
Encoded shape: (19071, 1883)

Running FP-Growth...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag


FP-Growth completed.
Number of frequent itemsets: 3973

Itemset size distribution:
itemsets
1    1883
2    2090
Name: count, dtype: int64


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [4]:
# ============================================================
# CELL 4 — GENERATE FP-GROWTH ASSOCIATION RULES
# ============================================================

from mlxtend.frequent_patterns import association_rules

rules_fpgrowth = association_rules(
    frequent_itemsets_fpgrowth,
    metric="confidence",
    min_threshold=0.10
)

print("FP-Growth association rules generated.")
print("Number of rules:", len(rules_fpgrowth))

print("\nTop 5 rules by lift:")
print(
    rules_fpgrowth[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ]
    .sort_values("lift", ascending=False)
    .head(5)
    .to_string(index=False)
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

FP-Growth association rules generated.
Number of rules: 1468

Top 5 rules by lift:
antecedents consequents  support  confidence       lift
    (15984)     (38312) 0.001258    0.685714 335.314286
    (38312)     (15984) 0.001258    0.615385 335.314286
    (38312)     (48220) 0.001363    0.666667 334.578947
    (48220)     (38312) 0.001363    0.684211 334.578947
    (15984)     (48220) 0.001049    0.571429 286.781955


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [6]:
# ============================================================
# CELL 5 — COMPARE APRIORI AND FP-GROWTH
# ============================================================

# Apriori results obtained before the runtime crash
apriori_itemsets = 3670
apriori_rules = 1261

# FP-Growth results from the current runtime
fpgrowth_itemsets = len(frequent_itemsets_fpgrowth)
fpgrowth_rules = len(rules_fpgrowth)

comparison = pd.DataFrame({
    "Method": ["Apriori", "FP-Growth"],
    "Frequent Itemsets": [
        apriori_itemsets,
        fpgrowth_itemsets
    ],
    "Association Rules": [
        apriori_rules,
        fpgrowth_rules
    ]
})

print("Apriori vs FP-Growth comparison:\n")
print(comparison.to_string(index=False))

Apriori vs FP-Growth comparison:

   Method  Frequent Itemsets  Association Rules
  Apriori               3670               1261
FP-Growth               3973               1468


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [7]:
# ============================================================
# CELL 6 — EXTRACT FP-GROWTH PRODUCT PAIR ASSOCIATIONS
# ============================================================

fp_pairs = frequent_itemsets_fpgrowth[
    frequent_itemsets_fpgrowth["itemsets"].apply(len) == 2
].copy()

print("FP-Growth product pairs extracted.")
print("Number of product pairs:", len(fp_pairs))

print("\nSample product pairs:")
print(
    fp_pairs[
        ["support", "itemsets"]
    ]
    .sort_values("support", ascending=False)
    .head(10)
    .to_string(index=False)
)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

FP-Growth product pairs extracted.
Number of product pairs: 2090

Sample product pairs:
 support       itemsets
0.021079 (13176, 21137)
0.019506 (13176, 47209)
0.018143 (21137, 24852)
0.016517 (24852, 47766)
0.016307 (24852, 21903)
0.014525 (13176, 21903)
0.014210 (47626, 24852)
0.013895 (24852, 16797)
0.013371 (47209, 21137)
0.013214 (21137, 21903)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [8]:
# ============================================================
# CELL 7 — CREATE COMPACT PRODUCT-PAIR ASSOCIATION TABLE
# ============================================================

pair_records = []

for _, row in fp_pairs.iterrows():
    items = list(row["itemsets"])

    product_a = items[0]
    product_b = items[1]
    support = row["support"]

    pair_records.append({
        "product_a": product_a,
        "product_b": product_b,
        "support": support
    })

association_pairs = pd.DataFrame(pair_records)

print("Compact association table created.")
print("Number of product pairs:", len(association_pairs))

print("\nColumns:")
print(association_pairs.columns.tolist())

print("\nSample:")
print(association_pairs.head(10).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Compact association table created.
Number of product pairs: 2090

Columns:
['product_a', 'product_b', 'support']

Sample:
 product_a  product_b  support
     24852      47766 0.016517
     47766      21903 0.009963
     13176      47766 0.007865
     21137      47766 0.008442
     24852      29487 0.004405
     47766      29487 0.002255
     26209      29487 0.001940
     13176      29487 0.001416
     21137      29487 0.001258
     31717      29487 0.001049


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [9]:
# ============================================================
# CELL 8 — CREATE BIDIRECTIONAL ASSOCIATION TABLE
# ============================================================

association_reverse = association_pairs.rename(
    columns={
        "product_a": "product_b",
        "product_b": "product_a"
    }
)

association_bidirectional = pd.concat(
    [
        association_pairs,
        association_reverse
    ],
    ignore_index=True
)

print("Bidirectional association table created.")
print("Total association entries:",
      len(association_bidirectional))

print("\nSample:")
print(
    association_bidirectional
    .head(10)
    .to_string(index=False)
)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Bidirectional association table created.
Total association entries: 4180

Sample:
 product_a  product_b  support
     24852      47766 0.016517
     47766      21903 0.009963
     13176      47766 0.007865
     21137      47766 0.008442
     24852      29487 0.004405
     47766      29487 0.002255
     26209      29487 0.001940
     13176      29487 0.001416
     21137      29487 0.001258
     31717      29487 0.001049


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [10]:
# ============================================================
# CELL 9 — CREATE ASSOCIATION SCORE LOOKUP
# ============================================================

association_score_lookup = (
    association_bidirectional
    .groupby(
        ["product_a", "product_b"]
    )["support"]
    .max()
    .to_dict()
)

print("Association score lookup created.")
print("Number of directional associations:",
      len(association_score_lookup))

# Show a few examples
print("\nSample association scores:")

sample_items = list(association_score_lookup.items())[:10]

for (product_a, product_b), score in sample_items:
    print(
        f"{product_a} -> {product_b}: "
        f"{score:.6f}"
    )


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Association score lookup created.
Number of directional associations: 4180

Sample association scores:
45 -> 21137: 0.001206
45 -> 24852: 0.001311
260 -> 13176: 0.001311
260 -> 21137: 0.001940
260 -> 24852: 0.001626
329 -> 13176: 0.001049
432 -> 13176: 0.001049
432 -> 21137: 0.001206
432 -> 21903: 0.001363
432 -> 24852: 0.002360


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [11]:
# ============================================================
# CELL 10 — SAVE ASSOCIATION ARTIFACTS
# ============================================================

ASSOCIATION_PATH = os.path.join(
    MODEL_PATH,
    "association_pairs_fpgrowth.csv"
)

SUMMARY_PATH = os.path.join(
    MODEL_PATH,
    "association_mining_summary.csv"
)

# Save bidirectional association pairs
association_bidirectional.to_csv(
    ASSOCIATION_PATH,
    index=False
)

# Save method comparison
association_summary = pd.DataFrame({
    "method": ["Apriori", "FP-Growth"],
    "frequent_itemsets": [3670, 3973],
    "association_rules": [1261, 1468]
})

association_summary.to_csv(
    SUMMARY_PATH,
    index=False
)

print("Association artifacts saved successfully.")
print("Pair file:", ASSOCIATION_PATH)
print("Summary file:", SUMMARY_PATH)
print("\nAssociation summary:")
print(association_summary.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Association artifacts saved successfully.
Pair file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/association_pairs_fpgrowth.csv
Summary file: /content/drive/MyDrive/ML_Market_Basket_Analysis/Models/Model_Outputs/association_mining_summary.csv

Association summary:
   method  frequent_itemsets  association_rules
  Apriori               3670               1261
FP-Growth               3973               1468


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [12]:
# ============================================================
# CELL 11 — TEMPORAL-SAFETY VALIDATION
# ============================================================

print("Temporal-safety validation")
print("-" * 40)

print("Maximum training order number:", TRAIN_ORDER_MAX)

print(
    "Association mining source:",
    "Historical orders with order_number <= 50"
)

print(
    "\nTemporal rule satisfied:",
    TRAIN_ORDER_MAX <= 50
)

print(
    "Validation/future orders used in mining:",
    "NO"
)

Temporal-safety validation
----------------------------------------
Maximum training order number: 50
Association mining source: Historical orders with order_number <= 50

Temporal rule satisfied: True
Validation/future orders used in mining: NO


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [13]:
# ============================================================
# CELL 12 — FINAL ARTIFACT VERIFICATION
# ============================================================

print("Final association artifact verification")
print("-" * 45)

print("Pair file exists:", os.path.exists(ASSOCIATION_PATH))
print("Summary file exists:", os.path.exists(SUMMARY_PATH))

# Load saved artifacts
saved_pairs = pd.read_csv(ASSOCIATION_PATH)
saved_summary = pd.read_csv(SUMMARY_PATH)

print("\nSaved pair artifact:")
print("Shape:", saved_pairs.shape)
print("Columns:", saved_pairs.columns.tolist())

print("\nSaved summary artifact:")
print("Shape:", saved_summary.shape)
print("Columns:", saved_summary.columns.tolist())

print("\nVerification completed successfully.")


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Final association artifact verification
---------------------------------------------
Pair file exists: True
Summary file exists: True

Saved pair artifact:
Shape: (4180, 3)
Columns: ['product_a', 'product_b', 'support']

Saved summary artifact:
Shape: (2, 3)
Columns: ['method', 'frequent_itemsets', 'association_rules']

Verification completed successfully.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag